# ntv_ddml

> 对应代码：`EconometricsCode/code_in_notes/Chap.10/ntv_ddml.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.10`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [1]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.10")

import stata_setup
stata_setup.config("/opt/Stata", "mp")


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Single-user 16-core  perpetual
Serial number: 501806307019
  Licensed to: Keep calm and DO your research
               Jichun Si

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


我们加载俄罗斯 NTV 的地区-年份面板数据，并用 tsset 声明面板维度（地区 tik_id 与时间 year），为随后生成滞后变量做准备。

In [2]:
%%stata
clear
set more off
use ../datasets/NTV_Aggregate_Data_reshaped.dta
tsset tik_id year


. clear

. set more off

. use ../datasets/NTV_Aggregate_Data_reshaped.dta

. tsset tik_id year

Panel variable: tik_id (strongly balanced)
 Time variable: year, 1995 to 2003, but with gaps
         Delta: 1 unit

. 


控制变量取护士数、医生数、人口与工资的一期滞后（L. 算子），滞后取值可以缓解同期反向因果的担忧。

In [3]:
%%stata
// 控制变量
gen lag_nurses=L.nurses
gen lag_doctors_pc=L.doctors_pc
gen lag_pop=L.logpop
gen lag_wage=L.log_wage


. // 控制变量
. gen lag_nurses=L.nurses
(9,744 missing values generated)

. gen lag_doctors_pc=L.doctors_pc
(9,744 missing values generated)

. gen lag_pop=L.logpop
(7,629 missing values generated)

. gen lag_wage=L.log_wage
(7,676 missing values generated)

. 


我们把四个滞后控制与地区虚拟变量 i.region 一并收入局部宏 controls，供后续两个条件期望方程共同使用。

In [4]:
%%stata
local controls "lag_doctors_pc lag_nurses lag_pop lag_wage i.region"

ddml init partial 初始化一个部分线性模型（partially linear model）：结果方程 Y=g(X)+θD+U，处理方程 D=m(X)+V，我们的目标参数是 θ。kfolds(10) 指定把样本随机分为 10 折做交叉拟合（cross-fitting），即每一折的残差化都只用其余 9 折训练出的模型完成，避免用自己的数据拟合自己所带来的过拟合偏差。

In [5]:
%%stata
// DDML
ddml init partial, kfolds(10)


. // DDML
. ddml init partial, kfolds(10)

. 


这一步为 E[Y|X] 指定机器学习算法：用 pystacked 调用随机森林（method(rf)，n_estimators(200) 即 200 棵树），以控制变量 X 预测结果变量 Votes_SPS_（支持 SPS 党的得票）。ddml 会据此在每一折外样本上生成 Y 的条件期望拟合值。

In [6]:
%%stata
ddml E[Y|X]: pystacked Votes_SPS_  `controls', type(regress) method(rf) cmdopt1(n_estimators(200))

Learner Y1_pystacked added successfully.


类似地，我们为 E[D|X] 指定随机森林回归：用同样的控制变量 X 预测处理变量 Watch_probit（地区收看 NTV 的强度）。对 Y 和 D 分别关于 X 残差化，正是 Frisch–Waugh–Lovell 思想在高维控制下的推广，也是双重机器学习"正交化"的关键一步。

In [7]:
%%stata
ddml E[D|X]: pystacked Watch_probit `controls', type(regress) method(rf) cmdopt1(n_estimators(200))

Learner D1_pystacked added successfully.


ddml crossfit 执行交叉拟合：在每一折的留一样本上计算残差 Y-E[Y|X] 与 D-E[D|X]，由于 Neyman 正交性，这些残差对机器学习第一步的估计误差不敏感，从而允许我们用非参数方法灵活拟合烦杂函数 g(·)、m(·)。

In [8]:
%%stata
ddml crossfit

Cross-fitting E[y|X] equation: Votes_SPS_
Cross-fitting fold 1 2 3 4 5 6 7 8 9 10 ...completed cross-fitting
Cross-fitting E[D|X] equation: Watch_probit
Cross-fitting fold 1 2 3 4 5 6 7 8 9 10 ...completed cross-fitting


最后用残差对残差做回归，得到处理效应 θ 的估计，并按地区聚类报告稳健标准误（cluster(region)）。这正等价于注释中所示的 reg Y1_pystacked_1 D1_pystacked_1——部分线性模型 DML 的第二阶段就是一个简单的一元回归。

In [9]:
%%stata
ddml estimate,  cluster(region)


DDML estimation results:
spec  r     Y learner     D learner         b        SE
 opt  1  Y1_pystacked  D1_pystacked     4.022  ( 1.213)
opt = minimum MSE specification for that resample.

Min MSE DDML model
y-E[y|X]  = Y1_pystacked_1                         Number of obs   =      2054
D-E[D|X,Z]= D1_pystacked_1
                                (Std. err. adjusted for 83 clusters in region)
------------------------------------------------------------------------------
             |               Robust
  Votes_SPS_ | Coefficient  std. err.      z    P>|z|     [95% conf. interval]
-------------+----------------------------------------------------------------
Watch_probit |   4.022435   1.213127     3.32   0.001      1.64475    6.400121
       _cons |   .0251208   .0769164     0.33   0.744    -.1256326    .1758742
------------------------------------------------------------------------------

